In [ ]:
# Install all missing dependencies
!pip install cleantext bert-score transformers accelerate sentencepiece vllm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 135.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/1

In [ ]:
import zipfile, os
with zipfile.ZipFile('BINF4008_HW2.zip', 'r') as z:
    z.extractall('.')
os.chdir('BINF4008_HW2')
!ls

configs  README.md  src


In [ ]:
# Remount drive to refresh
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

!ls /content/drive/MyDrive/BINF4008_HW2/

In [ ]:
# Check what's actually in the drive
!ls /content/drive/
!ls /content/drive/Shareddrives/
!find /content/drive/Shareddrives -name "*.csv" 2>/dev/null
!find /content/drive/Shareddrives -type d 2>/dev/null

In [ ]:
import yaml

with open('configs/chexpert.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['model']['engine'] = {
    'dtype': 'bfloat16',
    'gpu_memory_utilization': 0.90,
    'max_model_len': 5000
}
cfg['batch_size'] = 64

with open('configs/chexpert.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

print("✓ updated")
!cat configs/chexpert.yaml

✓ updated
batch_size: 64
dataset:
  base_dir: /content/drive/MyDrive/BINF4008_HW2/images
  name: chexpert
metadata: /content/drive/MyDrive/BINF4008_HW2/metadata.csv
model:
  engine:
    dtype: bfloat16
    gpu_memory_utilization: 0.9
    max_model_len: 5000
  model_id: google/medgemma-4b-it
  name: medgemma
prompt:
  system: You are an expert radiologist.
  user: Provide a description of the findings in the radiology image.


In [ ]:
cfg2 = {
    'dataset': {'name': 'chexpert'},
    'model': {'name': 'medgemma'},
    'splits': None,
    'metrics': {
        'labels_path': 'experiments/chexpert/medgemma/labels.csv',
        'predictions_path': 'experiments/chexpert/medgemma/predictions.jsonl',
        'output_dir': 'experiments/chexpert/medgemma/',
        'bertscore': {
            'model_type': 'distilbert-base-uncased',
            'lang': 'en',
            'batch_size': 16,
            'rescale_with_baseline': False
        }
    }
}

with open('configs/compute_metrics.yaml', 'w') as f:
    yaml.safe_dump(cfg2, f)

print("✓ compute_metrics.yaml written")
!cat configs/compute_metrics.yaml

✓ compute_metrics.yaml written
dataset:
  name: chexpert
metrics:
  bertscore:
    batch_size: 16
    lang: en
    model_type: distilbert-base-uncased
    rescale_with_baseline: false
  labels_path: experiments/chexpert/medgemma/labels.csv
  output_dir: experiments/chexpert/medgemma/
  predictions_path: experiments/chexpert/medgemma/predictions.jsonl
model:
  name: medgemma
splits: null


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Stage 1 — takes ~30 min, watch for progress bar
!python -m src.runners.run_eval --config configs/chexpert.yaml

[INFO] Preprocessed file not found, generating: gpt_processed_data.jsonl
/content/drive/MyDrive/BINF4008_HW2/metadata.csv
Number of null findings:  160
Number of null impressions:  0
Length of chexpert data after preprocessing:  234
split
valid    234
Name: count, dtype: int64
Generate_Method value counts in processed GPT data:  generate_method
gpt    234
Name: count, dtype: int64
[INFO] Wrote preprocessed records to: gpt_processed_data.jsonl
2026-03-16 16:17:11.280316: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 16:17:11.298779: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773677831.322988    4144 cuda_dnn.cc

In [38]:
!python -m src.runners.extract_chexpert_labels \
  --predictions experiments/chexpert/medgemma/predictions.jsonl \
  --ground-truth /content/drive/MyDrive/BINF4008_HW2/label.csv \
  --output-csv   experiments/chexpert/medgemma/labels.csv \
  --model-id     Qwen/Qwen2.5-3B-Instruct \
  --include-text

[INFO] Mapping summary: {'relative_path': 234, 'patient_study_view': 0, 'patient_study': 0, 'patient_only': 0, 'row_order_fallback': 0, 'unmatched': 0}
2026-03-16 17:07:13.384869: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 17:07:13.406500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773680833.427269   17728 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773680833.433630   17728 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been re

In [39]:
!python -m src.runners.compute_metrics --config configs/compute_metrics.yaml

[INFO] No splits path provided; skipping race/gender TPR.
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 350kB/s]
config.json: 100% 483/483 [00:00<00:00, 3.68MB/s]
vocab.txt: 232kB [00:00, 1.70MB/s]
tokenizer.json: 466kB [00:00, 1.64MB/s]
2026-03-16 17:10:26.559393: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 17:10:26.577757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773681026.600058   18616 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773681026.607493   18616 cuda_blas.cc:1407] Unable to register c

In [40]:
import pandas as pd

base = 'experiments/chexpert/medgemma/'

# BERTScore
bert = pd.read_csv(base + 'metrics_bertscore_summary.csv').iloc[0]
print('='*55)
print('BERTScore (distilbert-base-uncased)')
print('='*55)
print(f'  Precision : {bert["bertscore_precision_mean"]:.4f}')
print(f'  Recall    : {bert["bertscore_recall_mean"]:.4f}')
print(f'  F1        : {bert["bertscore_f1_mean"]:.4f}')

# Per-disease F1
f1 = pd.read_csv(base + 'metrics_f1_by_disease.csv')
print()
print('='*55)
print('Per-Disease Classification F1')
print('='*55)
print(f'{"Disease":<35} {"F1":>8} {"Support":>8}')
print('-'*55)
for _, row in f1.iterrows():
    f1v = f'{row["f1"]:.4f}' if str(row["f1"]) not in ('nan','None') else '  None'
    print(f'{row["label"]:<35} {f1v:>8} {int(row["support"]):>8}')

# Aggregate
agg = pd.read_csv(base + 'metrics_f1_summary.csv').iloc[0]
print()
print('='*55)
print('Aggregate')
print('='*55)
print(f'  Macro-F1       : {agg["macro_f1"]:.4f}')
print(f'  Micro-F1       : {agg["micro_f1"]:.4f}')
print(f'  Micro-Precision: {agg["micro_precision"]:.4f}')
print(f'  Micro-Recall   : {agg["micro_recall"]:.4f}')

BERTScore (distilbert-base-uncased)
  Precision : 0.6745
  Recall    : 0.7900
  F1        : 0.7273

Per-Disease Classification F1
Disease                                   F1  Support
-------------------------------------------------------
No Finding                            0.3645       38
Enlarged Cardiomediastinum            0.3308      109
Cardiomegaly                          0.5098       68
Lung Opacity                          0.6129      126
Lung Lesion                           0.0000        1
Edema                                 0.5000       45
Consolidation                         0.3956       33
Pneumonia                             0.3243        8
Atelectasis                           0.3306       80
Pneumothorax                          0.2727        8
Pleural Effusion                      0.7130       67
Pleural Other                         0.0000        1
Fracture                                None        0
Support Devices                       0.7983      107

Agg

In [41]:
from google.colab import files
import shutil

shutil.make_archive('/content/chexpert_results', 'zip',
                    '/content/BINF4008_HW2/experiments')
files.download('/content/chexpert_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
import yaml

with open('configs/chexpert.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['prompt']['system'] = (
    "You are an expert radiologist specializing in chest X-ray interpretation. "
    "You write concise, structured radiology reports in standard clinical format."
)

cfg['prompt']['user'] = (
    "Analyze this chest X-ray and write a radiology report with two sections:\n\n"
    "FINDINGS: Describe what you observe regarding the lungs, pleura, cardiac silhouette, "
    "mediastinum, and any support devices. Be specific about location, size, and severity.\n\n"
    "IMPRESSION: Summarize the key findings as a concise diagnostic conclusion. "
    "Explicitly state if any of the following are present or absent: "
    "pleural effusion, cardiomegaly, pulmonary edema, consolidation, atelectasis, "
    "pneumothorax, lung opacity, support devices, fracture.\n\n"
    "If the image appears normal, state 'No acute cardiopulmonary findings.'"
)

with open('configs/chexpert.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

print("✓ prompt updated")
!cat configs/chexpert.yaml

✓ prompt updated
batch_size: 64
dataset:
  base_dir: /content/drive/MyDrive/BINF4008_HW2/images
  name: chexpert
metadata: /content/drive/MyDrive/BINF4008_HW2/metadata.csv
model:
  engine:
    dtype: bfloat16
    gpu_memory_utilization: 0.9
    max_model_len: 5000
  model_id: google/medgemma-4b-it
  name: medgemma
prompt:
  system: You are an expert radiologist specializing in chest X-ray interpretation.
    You write concise, structured radiology reports in standard clinical format.
  user: 'Analyze this chest X-ray and write a radiology report with two sections:


    FINDINGS: Describe what you observe regarding the lungs, pleura, cardiac silhouette,
    mediastinum, and any support devices. Be specific about location, size, and severity.


    IMPRESSION: Summarize the key findings as a concise diagnostic conclusion. Explicitly
    state if any of the following are present or absent: pleural effusion, cardiomegaly,
    pulmonary edema, consolidation, atelectasis, pneumothorax, lung

In [44]:
# delete the old predictions so Stage 1 reruns
import os
for f in ['gpt_processed_data.jsonl',
          'experiments/chexpert/medgemma/predictions.jsonl',
          'experiments/chexpert/medgemma/labels.csv']:
    if os.path.exists(f):
        os.remove(f)
        print(f"✓ deleted {f}")

✓ deleted gpt_processed_data.jsonl
✓ deleted experiments/chexpert/medgemma/predictions.jsonl
✓ deleted experiments/chexpert/medgemma/labels.csv


In [45]:
# Stage 1
!python -m src.runners.run_eval --config configs/chexpert.yaml

[INFO] Preprocessed file not found, generating: gpt_processed_data.jsonl
/content/drive/MyDrive/BINF4008_HW2/metadata.csv
Number of null findings:  160
Number of null impressions:  0
Length of chexpert data after preprocessing:  234
split
valid    234
Name: count, dtype: int64
Generate_Method value counts in processed GPT data:  generate_method
gpt    234
Name: count, dtype: int64
[INFO] Wrote preprocessed records to: gpt_processed_data.jsonl
2026-03-16 18:10:53.260120: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 18:10:53.279354: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773684653.303680   33898 cuda_dnn.cc

In [50]:
# Stage 2
!python -m src.runners.extract_chexpert_labels \
  --predictions experiments/chexpert/medgemma/predictions.jsonl \
  --ground-truth /content/drive/MyDrive/BINF4008_HW2/label.csv \
  --output-csv   experiments/chexpert/medgemma/labels.csv \
  --model-id     Qwen/Qwen2.5-3B-Instruct \
  --include-text

[INFO] Mapping summary: {'relative_path': 234, 'patient_study_view': 0, 'patient_study': 0, 'patient_only': 0, 'row_order_fallback': 0, 'unmatched': 0}
2026-03-16 18:22:23.780181: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 18:22:23.801538: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773685343.822096   37213 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773685343.828408   37213 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been re

In [51]:
# Stage 3
!python -m src.runners.compute_metrics --config configs/compute_metrics.yaml

[INFO] No splits path provided; skipping race/gender TPR.
2026-03-16 18:24:36.198217: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 18:24:36.217018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773685476.239495   37859 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773685476.246864   37859 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773685476.266314   37859 computation_placer.cc:177] computation placer a

In [52]:
import pandas as pd
base = 'experiments/chexpert/medgemma/'

bert = pd.read_csv(base + 'metrics_bertscore_summary.csv').iloc[0]
print('='*55)
print('BERTScore IMPROVED')
print('='*55)
print(f'  Precision : {bert["bertscore_precision_mean"]:.4f}')
print(f'  Recall    : {bert["bertscore_recall_mean"]:.4f}')
print(f'  F1        : {bert["bertscore_f1_mean"]:.4f}')

f1 = pd.read_csv(base + 'metrics_f1_by_disease.csv')
print()
print('='*55)
print('Per-Disease F1 IMPROVED')
print('='*55)
print(f'{"Disease":<35} {"F1":>8} {"Support":>8}')
print('-'*55)
for _, row in f1.iterrows():
    f1v = f'{row["f1"]:.4f}' if str(row["f1"]) not in ('nan','None') else '  None'
    print(f'{row["label"]:<35} {f1v:>8} {int(row["support"]):>8}')

agg = pd.read_csv(base + 'metrics_f1_summary.csv').iloc[0]
print()
print(f'  Macro-F1 : {agg["macro_f1"]:.4f}')
print(f'  Micro-F1 : {agg["micro_f1"]:.4f}')

BERTScore IMPROVED
  Precision : 0.7386
  Recall    : 0.8086
  F1        : 0.7714

Per-Disease F1 IMPROVED
Disease                                   F1  Support
-------------------------------------------------------
No Finding                            0.3089       38
Enlarged Cardiomediastinum            0.5166      109
Cardiomegaly                          0.4130       68
Lung Opacity                          0.4881      126
Lung Lesion                           0.0000        1
Edema                                 0.3103       45
Consolidation                         0.4286       33
Pneumonia                             0.0000        8
Atelectasis                           0.3925       80
Pneumothorax                          0.0000        8
Pleural Effusion                      0.6964       67
Pleural Other                           None        1
Fracture                                None        0
Support Devices                       0.6734      107

  Macro-F1 : 0.3523
  Micr